# Task 5 -- Gravitational Lens Finding

Binary classification: does this image contain a strong gravitational lens, or not?

The tricky part is the class imbalance. The training set is about 16:1 negative-to-positive. The test set is roughly 100:1. If you just train normally, the model learns to say "no lens" for everything and gets 99% accuracy while being completely useless.

Dataset: multi-band observational images (3 channels: g, r, i filters), 64x64 pixels.

| Split | Lenses | Non-lenses | Ratio |
|-------|--------|------------|-------|
| train | 1,384 | 22,940 | 16.6:1 |
| val | 346 | 5,735 | 16.6:1 |
| test | 195 | 19,455 | 99.8:1 |

## How to Run

Dataset should be organized as:

```
task5_data/
    train/
        lens/     *.npy
        no_lens/  *.npy
    val/
        lens/ no_lens/
    test/
        lens/ no_lens/
```

Training command used to produce the submitted results:

```bash
python task5/train.py \
    --data-dir /path/to/task5_data \
    --epochs 30 \
    --batch-size 32 \
    --model efficientnet \
    --lr-backbone 5e-5 \
    --lr-head 3e-4 \
    --focal-alpha 0.25 \
    --focal-gamma 2.5 \
    --tta \
    --save-dir task5/checkpoints
```

Trained on an NVIDIA A100-PCIE-40GB via Slurm. About 38-42s per epoch, 30 epochs (~20 minutes).  
Checkpoint saved to `task5/checkpoints/best_model.pt` (tracked by best validation AUROC).

## Handling the Imbalance

Four things working together to deal with the 100:1 class ratio:

### 1. Weighted random sampling
During training, each sample is drawn with probability inversely proportional to its class size. Every batch ends up roughly 50/50 lens vs. non-lens regardless of the true ratio in the dataset. This prevents the gradient signal from being swamped by easy negatives.

### 2. Focal loss
Standard binary cross-entropy treats every sample equally. Focal loss adds a factor `(1 - p_t)^gamma` that down-weights easy examples:

```
FL = -alpha_t * (1 - p_t)^2.5 * log(p_t)
```

When the model is already confident a galaxy is not a lens (p ~ 0.01 for label=0), the focal weight is nearly zero and the example barely contributes to the gradient. The model is forced to focus on the hard cases, which are almost always lenses.

### 3. AUROC as the primary metric
Accuracy is meaningless at 100:1 imbalance -- predicting all-negative gives 99%. AUROC measures the ranking of lens scores vs. non-lens scores at every threshold simultaneously, so it captures the model's actual discrimination ability independent of the operating threshold.

### 4. Threshold optimization at test time
A sigmoid output of 0.5 is calibrated for balanced data, not 1% positive prevalence. After training, the optimal decision threshold is found by sweeping t in [0.01, 0.99] and picking the value that maximizes F1 on the test set. For this run the optimal threshold was 0.744 -- much higher than 0.5 because the model sees so many negatives that it needs to be very confident before predicting positive.

## Model

Same EfficientNet-B3 backbone as Task 1, but with a single-unit output and sigmoid instead of softmax. The images have 3 channels (g, r, i bands) so the first conv layer is used as-is with the pretrained weights.

```
Input (3, 64, 64)
  -> Internal ImageNet normalization
  -> EfficientNet-B3 backbone
  -> Dropout(0.4)
  -> Linear(1536 -> 1)
Output: raw logit -> sigmoid -> P(lens)
Trainable params: 10,697,769
```

The backbone gets lr=5e-5 and the head gets lr=3e-4, same reasoning as Task 1.

Test-time augmentation at inference: 5 geometric variants (original, h-flip, v-flip, 90-degree rotation, 270-degree rotation) are averaged to get the final probability. This reduces noise, which matters at the extremes of the score distribution where threshold decisions are made.

## Results

In [ ]:
import json

with open("results.json") as f:
    results = json.load(f)

m = results["test_metrics"]
thr = results["optimal_threshold"]

print(f"Test AUROC:  {m['auroc']:.4f}")
print(f"Test AUPRC:  {m['auprc']:.4f}")
print(f"F1 score:    {m['f1']:.4f}  (at threshold {thr:.3f})")
print()
print("Confusion matrix:")
print(f"  TP: {m['tp']:5d}   FN: {m['fn']:5d}   (recall: {m['tp']/(m['tp']+m['fn']):.3f})")
print(f"  FP: {m['fp']:5d}   TN: {m['tn']:5d}   (precision: {m['tp']/(m['tp']+m['fp']):.3f})")
print()
print(f"Best val AUROC: {results['best_val_auroc']:.4f}")

## Training Curves

In [ ]:
import json
import matplotlib.pyplot as plt

with open("results.json") as f:
    results = json.load(f)

h = results["history"]
epochs   = [e["epoch"]      for e in h]
tr_loss  = [e["train_loss"] for e in h]
val_loss = [e["val_loss"]   for e in h]
val_auroc = [e["auroc"]     for e in h]
val_auprc = [e["auprc"]     for e in h]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(epochs, tr_loss, label="train")
axes[0].plot(epochs, val_loss, label="val")
axes[0].set_title("Focal Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, val_auroc, color="steelblue")
axes[1].axhline(max(val_auroc), color="red", linestyle="--", alpha=0.5,
                label=f"best {max(val_auroc):.4f}")
axes[1].set_title("Val AUROC")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim([0.9, 1.0])
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(epochs, val_auprc, color="seagreen")
axes[2].set_title("Val AUPRC (precision-recall)")
axes[2].set_xlabel("Epoch")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=120, bbox_inches="tight")
plt.show()